# Notebook 03: SHAP background-sample check (decision I-16, R1.3)

Reviewer 1 (comment 3) notes that summed SHAP attributions may depend on the selected background sample. This
notebook repeats the attribution of notebook 02 _r1_ext for Delhi, Mumbai and Bengaluru against two new background
samples of 50 training windows each, drawn outside the original background. Model, explained test windows, method
(one value per channel, no limit) and KernelSHAP seed are those of notebook 02 _r1_ext, so only the background
changes. It first recomputes 10 windows with the original background to show they equal the saved values.

It reads notebook 02 _r1_ext's folder `MyDrive/phase4_v2_results_r1_ext` (model, dataset, SHAP caches) and writes only
to `reports/shap/bg_check/` in it. No decomposition and no training. Runtime: GPU (T4), about 1.5 hours.

Run order of the repository: `01_phase1_baselines_r1`, `02_ceemdan_itransformer_shap_r1`, `02_ceemdan_itransformer_shap_r1_ext`, `03_shap_background_check_r1_ext`. The README explains the decision tags used in comments and printed output (for example R1.13 or I-15).

In [6]:
!nvidia-smi -L
!pip install -q "shap==0.51.0"

GPU 0: Tesla T4 (UUID: GPU-dd11c3df-a85f-1241-0012-39297c429a66)


In [7]:
import sys
import time
import json
import pickle
import random
import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import matplotlib
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('phase4_v2_notebook')

print(f"torch version   : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

torch version   : 2.11.0+cu130
CUDA available  : True


In [8]:
# ── GPU detection (KNOWN FACT #1) ───────────────────────────────────────────
# CEEMDANFeatureTrainer / iTransformerTrainer / GroupLSTMTrainer all default to
# device='cpu' in the original .py files and none of the original train_*.py scripts
# ever override this. We detect the device once, here, and pass it explicitly into
# every trainer constructor throughout this notebook.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cpu':
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU.")

Using device: cuda


In [9]:
RANDOM_SEED = 123  # from config_phase4_v2.py

def set_all_seeds(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Deterministic cuDNN kernels — slightly slower, but removes one more source of
    # run-to-run variance on GPU. Not present in the original (CPU-only) code, added here
    # since this notebook is GPU-specific.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(RANDOM_SEED)
print(f"All RNGs seeded with {RANDOM_SEED}")

All RNGs seeded with 123


In [10]:
# ── Revision 1, I-16: paths and constants. Reads the AR-extended run (I-15), writes only to reports/shap/bg_check ──
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = Path('/content/drive/MyDrive/phase4_v2_results_r1_ext')   # the I-15 run, nothing in it is changed
MODELS_DIR, REPORTS_DIR = RESULTS_DIR / 'models', RESULTS_DIR / 'reports'
SHAP_REPORTS_DIR = REPORTS_DIR / 'shap'   # the SHAP helper cell below reads its caches from shap_cache_r1 here
# The same values as in notebook 02 _r1_ext (Sections 3 and 8b), used by the SHAP helper cell
R19_VAL_START = np.datetime64('2019-04-01')
R19_TEST_START = np.datetime64('2019-07-01')
R110_LOCKDOWN_START = np.datetime64('2020-03-25')
for p in (MODELS_DIR, SHAP_REPORTS_DIR / 'shap_cache_r1'):
    assert p.is_dir(), f'{p} not found: the I-15 run must be on Drive'
print(f'Reading the I-15 run from {RESULTS_DIR}')


Mounted at /content/drive
Reading the I-15 run from /content/drive/MyDrive/phase4_v2_results_r1_ext


In [11]:
class iTransformerModel(nn.Module):
    '''Transcribed from models/itransformer.py::iTransformerModel (unchanged).
    Revision 1, R1.16 (iv): forward takes an optional per-sample vector added to every channel token before the
    Transformer layers. It is None by default, so the plain iTransformer and the main model are unchanged.'''

    def __init__(self, seq_len, n_features=12, d_model=64, nhead=4,
                 ff_dim=256, n_layers=2, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.d_model = d_model

        # Project each variate's sequence to d_model
        self.variate_projection = nn.Linear(seq_len, d_model)

        # Learnable positional encoding over feature tokens
        self.pos_embedding = nn.Parameter(torch.zeros(1, n_features, d_model))
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        # Transformer over pollutant tokens (not time steps)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers, enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, x, token_offset=None):
        # x: (batch, seq_len, n_features)
        x = x.permute(0, 2, 1)                  # (batch, n_features, seq_len)
        x = self.variate_projection(x)           # (batch, n_features, d_model)
        x = x + self.pos_embedding
        if token_offset is not None:             # Revision 1, R1.16 (iv): (batch, d_model), same for every token
            x = x + token_offset.unsqueeze(1)
        x = self.transformer(x)                  # (batch, n_features, d_model)
        x = x.mean(dim=1)                        # (batch, d_model)
        x = self.norm(x)
        x = self.dropout(x)
        return self.output_proj(x)               # (batch, 1)

In [12]:
class CEEMDANFeatureITransformer(nn.Module):
    '''
    Transcribed from models/ceemdan_itransformer_v2.py::CEEMDANFeatureITransformer (unchanged).

    v2 model: iTransformer with CEEMDAN-augmented inputs + city bias correction.

    Input:  (batch, seq_len, n_features)  — Revision 1: 12 pollutants + 5 decomposition channels in sifting order (17);
            11 + 5 for the PM2.5 check (submitted: 12 + 6 energy-sorted IMFs = 18)
    Output: (batch, 1)                       — direct AQI prediction (scaled)

    The city_bias Embedding provides a learnable per-city AQI offset that
    accounts for systematic differences between Delhi (~300), Bangalore (~60), etc.
    Initialised at zero so the base iTransformer handles all cities identically
    at the start of training, then the bias adapts during backprop.
    '''

    # Revision 1, R1.16 (decision B1, nested variants): how the city enters the model.
    #   'none'        (i)   no city term (city_bias is created, so the shared weights start as in (ii), but unused)
    #   'bias'        (ii)  one learned number per city added to the prediction (the submitted model, the default)
    #   'scale_bias'  (iii) (ii) plus a per-city output scale: base * (1 + scale_c) + bias_c
    #   'vector_bias' (iv)  (ii) plus a learned city vector added to every channel token inside the Transformer
    # The added parts start at zero and are created without drawing random numbers, so with the same seed every
    # variant starts as exactly the same model and sees the same mini-batches.
    CONDITIONINGS = ('none', 'bias', 'scale_bias', 'vector_bias')

    def __init__(self, n_features=18, seq_len=14,
                 d_model=128, nhead=8, ff_dim=512, n_layers=4,
                 dropout=0.1, n_cities=26, conditioning='bias'):
        super().__init__()
        if conditioning not in self.CONDITIONINGS:
            raise ValueError(f'R1.16: conditioning must be one of {self.CONDITIONINGS}, not {conditioning!r}')
        self.conditioning = conditioning
        self.base = iTransformerModel(
            seq_len=seq_len, n_features=n_features,
            d_model=d_model, nhead=nhead, ff_dim=ff_dim,
            n_layers=n_layers, dropout=dropout,
        )
        self.city_bias = nn.Embedding(n_cities, 1)
        nn.init.zeros_(self.city_bias.weight)
        if conditioning == 'scale_bias':
            self.city_scale = nn.Parameter(torch.zeros(n_cities, 1))
        if conditioning == 'vector_bias':
            self.city_vector = nn.Parameter(torch.zeros(n_cities, d_model))

    def forward(self, x, city_id=None):
        '''
        Args:
            x:       (batch, seq_len, n_features)
            city_id: (batch,) LongTensor of city indices, or None
        Returns:
            (batch, 1) scaled AQI prediction
        '''
        use_city = city_id is not None and self.conditioning != 'none'
        offset = self.city_vector[city_id] if use_city and self.conditioning == 'vector_bias' else None
        pred = self.base(x, offset)                  # (batch, 1)
        if use_city:
            if self.conditioning == 'scale_bias':
                pred = pred * (1.0 + self.city_scale[city_id])
            pred = pred + self.city_bias(city_id)    # broadcast (batch, 1)
        return pred


class CEEMDANFeatureTrainer:
    '''
    Transcribed from models/ceemdan_itransformer_v2.py::CEEMDANFeatureTrainer (unchanged).

    Wraps CEEMDANFeatureITransformer with training, early stopping, save/load.

    Uses linear LR warmup followed by cosine decay — avoids the instability
    of pure cosine scheduling in the first few epochs when gradients are large.
    '''

    def __init__(self, seq_len, n_features, d_model, nhead, ff_dim, n_layers,
                 dropout, lr, weight_decay, n_cities, warmup_epochs=10,
                 device='cpu', conditioning='bias'):
        self.device = torch.device(device)
        self.lr = lr
        self.warmup_epochs = warmup_epochs
        self.config = {
            'seq_len': seq_len, 'n_features': n_features,
            'd_model': d_model, 'nhead': nhead, 'ff_dim': ff_dim,
            'n_layers': n_layers, 'dropout': dropout,
            'n_cities': n_cities, 'warmup_epochs': warmup_epochs,
            'conditioning': conditioning,   # Revision 1, R1.16; a saved model without it is 'bias'
        }
        self.model = CEEMDANFeatureITransformer(
            n_features=n_features, seq_len=seq_len,
            d_model=d_model, nhead=nhead, ff_dim=ff_dim,
            n_layers=n_layers, dropout=dropout, n_cities=n_cities,
            conditioning=conditioning,
        ).to(self.device)
        self.optimizer = Adam(
            self.model.parameters(), lr=lr, weight_decay=weight_decay
        )
        self.criterion = nn.MSELoss()

    # ── LR schedule ─────────────────────────────────────────────────────────

    def _compute_lr(self, epoch, total_epochs):
        '''Linear warmup then cosine decay.'''
        if epoch < self.warmup_epochs:
            return self.lr * (epoch + 1) / max(1, self.warmup_epochs)
        progress = (epoch - self.warmup_epochs) / max(
            1, total_epochs - self.warmup_epochs
        )
        return self.lr * 0.5 * (1.0 + np.cos(np.pi * progress))

    def _set_lr(self, lr):
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr

    # ── Training ─────────────────────────────────────────────────────────────

    def fit(self, X_train, y_train, city_train,
            X_val, y_val, city_val,
            epochs=150, batch_size=32, patience=15):
        '''
        Train with early stopping on validation MSE loss.

        Args:
            X_train:    (n_train, seq_len, n_features) float32
            y_train:    (n_train,) float32 — scaled AQI
            city_train: (n_train,) int64  — city index
            X_val:      (n_val, seq_len, n_features) float32
            y_val:      (n_val,) float32
            city_val:   (n_val,) int64
            epochs:     Max training epochs
            batch_size: Mini-batch size
            patience:   Early stopping patience (epochs without val improvement)
        '''
        X_tr = torch.FloatTensor(X_train).to(self.device)
        y_tr = torch.FloatTensor(y_train.reshape(-1, 1)).to(self.device)
        c_tr = torch.LongTensor(city_train).to(self.device)

        X_vl = torch.FloatTensor(X_val).to(self.device)
        y_vl = torch.FloatTensor(y_val.reshape(-1, 1)).to(self.device)
        c_vl = torch.LongTensor(city_val).to(self.device)

        dl = DataLoader(
            TensorDataset(X_tr, y_tr, c_tr),
            batch_size=batch_size, shuffle=True
        )

        best_val_loss = float('inf')
        patience_count = 0
        best_weights = None

        for epoch in range(1, epochs + 1):
            self._set_lr(self._compute_lr(epoch - 1, epochs))

            self.model.train()
            for X_b, y_b, c_b in dl:
                self.optimizer.zero_grad()
                pred = self.model(X_b, c_b)
                loss = self.criterion(pred, y_b)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.optimizer.step()

            self.model.eval()
            with torch.no_grad():
                val_pred = self.model(X_vl, c_vl)
                val_loss = self.criterion(val_pred, y_vl).item()

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_count = 0
                best_weights = {
                    k: v.clone() for k, v in self.model.state_dict().items()
                }
            else:
                patience_count += 1
                if patience_count >= patience:
                    logger.info(
                        f"  Early stop at epoch {epoch} "
                        f"(best_val_loss={best_val_loss:.6f})"
                    )
                    break

            if epoch % 25 == 0 or epoch == 1:
                current_lr = self.optimizer.param_groups[0]['lr']
                logger.info(
                    f"  Epoch {epoch:3d}/{epochs}: "
                    f"val_loss={val_loss:.6f}, lr={current_lr:.6f}"
                )

        if best_weights is not None:
            self.model.load_state_dict(best_weights)

        logger.info(f"  Training done. Best val_loss={best_val_loss:.6f}")

    # ── Inference ────────────────────────────────────────────────────────────

    def predict(self, X, city_ids=None):
        '''
        Args:
            X:        (n, seq_len, n_features) float32 array
            city_ids: (n,) int array of city indices, or None
        Returns:
            (n,) float32 array — scaled AQI predictions
        '''
        self.model.eval()
        with torch.no_grad():
            X_t = torch.FloatTensor(X).to(self.device)
            c_t = (
                torch.LongTensor(city_ids).to(self.device)
                if city_ids is not None else None
            )
            return self.model(X_t, c_t).cpu().numpy().flatten()

    # ── Persistence ──────────────────────────────────────────────────────────

    def save(self, path):
        '''Save model weights + config to a single .pt file.'''
        path = Path(path)
        torch.save({
            'model_state': self.model.state_dict(),
            'config': self.config,
        }, path)
        logger.info(f"  Model saved to {path}")

    @classmethod
    def load(cls, path, device='cpu'):
        '''Load a saved trainer. lr and weight_decay are placeholders (not used post-load).'''
        ckpt = torch.load(path, map_location=device)
        cfg = ckpt['config']
        trainer = cls(lr=1e-3, weight_decay=1e-5, device=device, **cfg)
        trainer.model.load_state_dict(ckpt['model_state'])
        trainer.model.eval()
        return trainer

In [13]:
# ── Revision 1, N2 (decision Q1): SHAP explains the model on exactly its training inputs ──
# Training saves its dataset next to the model. SHAP reloads it and takes one city's slice,
# instead of rebuilding that city alone (which gave per-city scalers and a different CEEMDAN seed).
N2_PRED_ATOL = 1e-4   # scaled-AQI units. Allows only float differences between devices


def save_training_dataset(path, data, y_pred_test_sc):
    '''Save the windows, city indices, test predictions and per-window channel periods of one trained model.'''
    # Revision 1, I-5: per window the periods of its decomposition's channels and the date of its last input day
    periods = {f'periods__{c}': a for c, a in data.get('city_periods', {}).items()}
    end_dates = {f'end_dates__{c}': np.asarray(a, dtype='datetime64[D]') for c, a in data.get('city_end_dates', {}).items()}
    np.savez_compressed(
        path,
        X_train=data['X_train'], city_train=data['city_train'],
        X_test=data['X_test'], city_test=data['city_test'], y_test=data['y_test'],
        y_pred_test_sc=np.asarray(y_pred_test_sc, dtype=np.float32),
        valid_cities=np.array(data['valid_cities']),
        **periods, **end_dates,
        # Revision 1, step 6d: target date and D1-filled flag of each test window (R1.9-4, R3.3 S1)
        **{k: data[k] for k in ('test_dates', 'test_filled') if k in data},
    )
    logger.info(f"  Training dataset saved to {path}")


def load_city_windows(path, city, city_to_idx, trainer):
    '''One city's training and test windows from a saved training dataset.

    Guards: the saved city order must match the model's city_to_idx, and the model must
    reproduce the test predictions saved at training time. Raises on any mismatch.
    '''
    saved = np.load(path, allow_pickle=False)
    saved_idx = {str(c): i for i, c in enumerate(saved['valid_cities'])}
    if saved_idx != dict(city_to_idx):
        raise ValueError(f"N2 guard: city indices in {path} do not match the model's city_to_idx")
    idx = city_to_idx[city]
    in_train = saved['city_train'] == idx
    in_test = saved['city_test'] == idx
    X_train_c, X_test_c = saved['X_train'][in_train], saved['X_test'][in_test]
    max_diff = 0.0
    if len(X_test_c):
        pred = trainer.predict(X_test_c, np.full(len(X_test_c), idx, dtype=np.int64))
        max_diff = float(np.max(np.abs(pred - saved['y_pred_test_sc'][in_test])))
    if not max_diff <= N2_PRED_ATOL:
        raise ValueError(f"N2 guard: the model does not reproduce the saved test predictions for {city} "
                         f"(max diff {max_diff:.2e}). Model and dataset come from different runs.")
    logger.info(f"  N2 guard passed for {city}: max prediction diff {max_diff:.1e}")
    return X_train_c, X_test_c


In [14]:
# ── Revision 1, R1.13 + N5 (step 6c): which windows SHAP explains, seeded, the same target dates for every model ──
# Decided 1 to 3 Oct 2026:
#   * Per city one seed, RANDOM_SEED + the city's index in C5_ALL_CITIES, for the test sample, the background sample
#     (two separate streams of it) and KernelSHAP's own sampling (np.random.seed right before it runs).
#   * The test target dates are drawn once per city from the main model's test windows (AQI, MAIN_LOOKBACK): the
#     lockdown/Unlock period (targets from R110_LOCKDOWN_START) in proportion to its share, rounded half up, the rest
#     pre-lockdown. The background target dates are drawn once per city from the main model's training windows.
#   * Every model (main, robustness lookback, ablation, PM2.5) explains its own windows forecasting those dates.
#     Sampled dates a model has no window for are counted and saved.
#   * Results are cached per city and run with the per-window values, and every field is checked on loading.
# Submitted: np.random.choice from the global random state, a separate draw per model, KernelSHAP unseeded (N5).
# Revision 1, I-15 (decided 6 Oct 2026, from I-13): KernelSHAP explains each channel as one group, its lookback days
# replaced together (one Shapley value per channel, as Methods describe), with l1_reg=False (shap 0.51.0's default
# keeps 10 non-zero values per window). shap_kernel_channel is the channel_nocap method of Section 18 of the run of
# record (Section 18 of notebook 02 _r1). Values are saved as (n, 1, n_features), so every summary, plot and table below
# reads them unchanged. compute_kernel_shap (Section 9 above) is no longer called.
from shap.utils._legacy import DenseData

SHAP_METHOD = 'channel_nocap'
import hashlib

SHAP_SEED_BASE = RANDOM_SEED
SHAP_CACHE_DIR = SHAP_REPORTS_DIR / 'shap_cache_r1'
SHAP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SHAP_PERIOD_ROWS = {}   # (run, lookback, city) -> rows of shap_period_rows, written per run by run_shap_for_city


def shap_city_seed(city):
    return SHAP_SEED_BASE + C5_ALL_CITIES.index(city)


def shap_window_dates(path, city):
    '''Target dates of one city's training and test windows in a saved training dataset, in the order of the
    slices load_city_windows returns.'''
    with np.load(path, allow_pickle=False) as saved:
        if f'end_dates__{city}' not in saved.files:
            raise ValueError(f'{path} has no window dates for {city}')
        end = saved[f'end_dates__{city}']
        idx = [str(c) for c in saved['valid_cities']].index(city)
        n_train, n_test = int(np.sum(saved['city_train'] == idx)), int(np.sum(saved['city_test'] == idx))
    target = end + np.timedelta64(1, 'D')   # the window's target is the day after its last input day
    train, test = target[:n_train], target[len(target) - n_test:]
    if len(target) < n_train + n_test or (len(train) and train[-1] >= R19_VAL_START) or (len(test) and test[0] < R19_TEST_START):
        raise ValueError(f'{path}: the window dates of {city} do not match its training and test windows')
    return train, test


def shap_lockdown_count(n_sample, n_pre, n_lockdown):
    '''Lockdown/Unlock dates among n_sample: n_sample x their share, rounded half up (in integers, no float ties).'''
    n = n_pre + n_lockdown
    return (2 * n_sample * n_lockdown + n) // (2 * n)


def shap_draw_dates(train_dates, test_dates, seed, n_test, n_background):
    '''Sampled test and background target dates of one city, sorted. Two separate streams of the city seed.'''
    rng_test, rng_bg = [np.random.default_rng(s) for s in np.random.SeedSequence(seed).spawn(2)]
    test_dates, train_dates = np.asarray(test_dates, dtype='datetime64[D]'), np.asarray(train_dates, dtype='datetime64[D]')
    test = test_dates
    if len(test_dates) > n_test:
        pre, lock = test_dates[test_dates < R110_LOCKDOWN_START], test_dates[test_dates >= R110_LOCKDOWN_START]
        n_lock = shap_lockdown_count(n_test, len(pre), len(lock))
        test = np.concatenate([pre[rng_test.choice(len(pre), n_test - n_lock, replace=False)],
                               lock[rng_test.choice(len(lock), n_lock, replace=False)]])
    background = train_dates
    if len(train_dates) > n_background:
        background = train_dates[rng_bg.choice(len(train_dates), n_background, replace=False)]
    return np.sort(test), np.sort(background)


def shap_sample(city, n_test, n_background):
    '''The city's sampled test and background target dates, drawn from the main model's windows (AQI, MAIN_LOOKBACK).'''
    train, test = shap_window_dates(get_v2_dataset_path(MAIN_LOOKBACK), city)
    return shap_draw_dates(train, test, shap_city_seed(city), n_test, n_background)


def shap_cache_path(run, city, lookback, n_test, n_background):
    return SHAP_CACHE_DIR / f'{run}_{city}_lb{lookback}_n{n_test}_bg{n_background}_seed{shap_city_seed(city)}.npz'


def shap_file_sha256(path):
    with open(path, 'rb') as f:
        return hashlib.sha256(f.read()).hexdigest()


def shap_kernel_channel(predict_fn, X_bg, X_t, seed):
    '''KernelSHAP of the windows X_t (n, L, F) against the background windows X_bg, one group per channel, cap off.
    Returns the values (n, F), the expected value, the predictions of X_t and the latency per window (ms).'''
    import shap

    n_days, n_feat = X_t.shape[1:]

    def flat_predict(X_flat):
        out = [predict_fn(X_flat[j:j + 64].reshape(-1, n_days, n_feat)) for j in range(0, len(X_flat), 64)]
        return np.concatenate(out)

    bg, xt = X_bg.reshape(len(X_bg), -1), X_t.reshape(len(X_t), -1)
    data = DenseData(bg, [f'channel_{j}' for j in range(n_feat)],   # input (day d, channel j) is d * n_feat + j
                     [np.arange(n_feat * n_days).reshape(n_days, n_feat)[:, j] for j in range(n_feat)])
    np.random.seed(seed)   # KernelSHAP samples from numpy's global state (shap 0.51.0)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        explainer = shap.KernelExplainer(flat_predict, data)
        start = time.time()
        values = np.vstack([explainer.shap_values(xt[i:i + 10], nsamples='auto', l1_reg=False, silent=True)
                            for i in range(0, len(xt), 10)])
        latency_ms = (time.time() - start) * 1000 / max(len(xt), 1)
    return values, float(np.ravel(explainer.expected_value)[0]), flat_predict(xt), latency_ms


def shap_explain_city(run, city, lookback, trainer, city_to_idx, dataset_path, model_path, feature_names,
                      n_background, max_test_samples):
    '''KernelSHAP values of one city's windows that forecast the sampled target dates, cached and checked.

    Returns a dict: shap_values (n, lookback, n_features), test_dates and background_dates (those explained and
    used), missing_test and missing_background (sampled dates the model has no window for), latency_ms.
    '''
    sample_test, sample_bg = shap_sample(city, max_test_samples, n_background)
    settings = {'run': run, 'city': city, 'lookback': lookback, 'seed': shap_city_seed(city),
                'feature_names': np.array(feature_names), 'model_sha256': shap_file_sha256(model_path),
                'sample_test_dates': sample_test, 'sample_background_dates': sample_bg,
                'method': np.array(SHAP_METHOD), 'l1_reg': np.array('False')}   # I-15
    path = shap_cache_path(run, city, lookback, max_test_samples, n_background)
    if path.exists():
        with np.load(path, allow_pickle=False) as z:
            for k, v in settings.items():
                if not np.array_equal(z[k], v):
                    raise ValueError(f'SHAP cache {path.name}: {k} differs from this run (another model or sample). '
                                     f'Delete the file to recompute.')
            logger.info(f'  {city}: SHAP values from the cache ({path.name})')
            return {k: z[k] for k in ('shap_values', 'test_dates', 'background_dates', 'missing_test',
                                      'missing_background', 'latency_ms')}

    X_train_c, X_test_c = load_city_windows(dataset_path, city, city_to_idx, trainer)   # N2 guard
    train_dates, test_dates = shap_window_dates(dataset_path, city)
    if len(train_dates) != len(X_train_c) or len(test_dates) != len(X_test_c):
        raise ValueError(f'{dataset_path}: window dates and windows of {city} differ in number')
    in_test, in_bg = np.isin(test_dates, sample_test), np.isin(train_dates, sample_bg)
    missing_test, missing_bg = int(len(sample_test) - in_test.sum()), int(len(sample_bg) - in_bg.sum())
    if not in_test.any() or not in_bg.any():
        raise ValueError(f'{run}, {city}: no window forecasts the sampled test or background dates')
    if missing_test or missing_bg:
        logger.info(f'  {run}, {city}: {missing_test} of {len(sample_test)} test dates and {missing_bg} of '
                    f'{len(sample_bg)} background dates have no window in this model')

    city_idx = city_to_idx[city]

    def predict_fn(X):
        return trainer.predict(X, np.full(len(X), city_idx, dtype=np.int64))

    values, expected, fx, latency_ms = shap_kernel_channel(predict_fn, X_train_c[in_bg], X_test_c[in_test],
                                                           shap_city_seed(city))   # I-15
    result = {'shap_values': values[:, None, :],   # I-15: (n, 1, n_features), one value per channel
              'expected_value': np.float64(expected), 'prediction': fx,
              'efficiency_max_abs_error': np.float64(np.max(np.abs(values.sum(axis=1) + expected - fx))),
              'test_dates': test_dates[in_test], 'background_dates': train_dates[in_bg],
              'missing_test': np.int64(missing_test), 'missing_background': np.int64(missing_bg),
              'latency_ms': np.float64(latency_ms)}
    np.savez_compressed(path, **settings, **result)
    logger.info(f'  {city}: SHAP values cached ({path.name})')
    return result


def shap_period_rows(run, city, lookback, res, feature_names, n_pollutants):
    '''Mean |SHAP| per channel over all explained windows and per period (R1.10), with the share of the channels
    after the pollutants (the decomposition channels, or AQI_lag in the ablation).'''
    lock = res['test_dates'] >= R110_LOCKDOWN_START
    rows = []
    for period, m in (('all', np.ones_like(lock)), ('pre_lockdown', ~lock), ('lockdown_unlock', lock)):
        v = np.abs(res['shap_values'][m]).mean(axis=(0, 1)) if m.any() else np.full(len(feature_names), np.nan)
        rows.append({'run': run, 'city': city, 'lookback': lookback, 'period': period, 'n_windows': int(m.sum()),
                     'missing_test_dates': int(res['missing_test']), 'missing_background_dates': int(res['missing_background']),
                     'non_pollutant_share_pct': float(v[n_pollutants:].sum() / v.sum() * 100) if m.any() else np.nan,
                     **dict(zip(feature_names, v))})
    return rows


def shap_write_periods(run, lookback):
    '''One CSV per run and lookback with the period rows of every city explained so far.'''
    rows = [r for (r_, lb, _), rs in SHAP_PERIOD_ROWS.items() if r_ == run and lb == lookback for r in rs]
    path = SHAP_REPORTS_DIR / f'shap_periods_{run}_lb{lookback}.csv'
    pd.DataFrame(rows).to_csv(path, index=False)
    return path


In [15]:
# ── Revision 1, I-16 (R1.3 (c)): does the attribution depend on the SHAP background sample? ──
# The main model of the I-15 run (lookback 7) explains the same 100 test windows per city as Sections 9 and 14 of
# notebook 02 _r1_ext, with the same method (shap_kernel_channel: one value per channel, l1_reg=False) and the same
# KernelSHAP seed (the city seed), but against new background samples: BG_N training windows drawn per draw from the
# city's training windows that are not in the original background (np.random.SeedSequence([city seed, draw])). Draws
# of one city are independent of each other, so they may share dates where a city has few training windows (Mumbai).
# Before the new draws, the first BG_N_REPRO windows are recomputed with the original background: they must equal the
# saved values (the same RNG state, so only device float differences), which shows the change comes from the
# background alone. Every result is cached in BG_DIR with its settings and checked on loading.
# Cost (I-15 run): about 8.4 s per window on a T4, so about 14 min per city and draw, about 1.5 h in all.
import hashlib
from scipy.stats import spearmanr

BG_CITIES = ['Delhi', 'Mumbai', 'Bengaluru']
BG_LOOKBACK = 7
BG_N = 50          # as in the I-15 run
BG_N_TEST = 100    # as in the I-15 run
BG_DRAWS = (1, 2)
BG_N_REPRO = 10    # one KernelSHAP chunk of shap_kernel_channel
BG_DIR = SHAP_REPORTS_DIR / 'bg_check'
BG_DIR.mkdir(parents=True, exist_ok=True)
BG_MODEL = MODELS_DIR / f'ceemdan_it_v2_lb{BG_LOOKBACK}.pt'
BG_SCALERS = MODELS_DIR / f'ceemdan_it_v2_lb{BG_LOOKBACK}_scalers.pkl'
BG_DATASET = MODELS_DIR / f'ceemdan_it_v2_lb{BG_LOOKBACK}_dataset.npz'
BG_FEATURES = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene',
               'IMF1', 'IMF2', 'IMF3', 'IMF4', 'IMF5+ (slow)']
BG_N_POLLUTANTS = 12


def bg_main_result(city, model_sha):
    '''The I-15 run's SHAP cache of one city, after checking it was made with this model and method.'''
    paths = sorted(SHAP_CACHE_DIR.glob(f'v2_{city}_lb{BG_LOOKBACK}_n{BG_N_TEST}_bg{BG_N}_seed*.npz'))
    if len(paths) != 1:
        raise ValueError(f'I-16: expected one SHAP cache of {city} in {SHAP_CACHE_DIR}, found {len(paths)}')
    with np.load(paths[0], allow_pickle=False) as z:
        main = {k: z[k] for k in z.files}
    if (str(main['run']) != 'v2' or str(main['city']) != city or int(main['lookback']) != BG_LOOKBACK
            or str(main['model_sha256']) != model_sha or str(main['method']) != 'channel_nocap'
            or str(main['l1_reg']) != 'False' or list(main['feature_names']) != BG_FEATURES
            or int(main['missing_test']) or int(main['missing_background'])):
        raise ValueError(f'I-16: {paths[0].name} was not made with {BG_MODEL.name} and the channel_nocap method')
    return main


def bg_draw(train_dates, original, city_seed, draw):
    '''BG_N target dates from the training windows outside the original background, sorted.'''
    pool = train_dates[~np.isin(train_dates, original)]
    if len(pool) < BG_N:
        raise ValueError(f'I-16: only {len(pool)} training windows outside the original background')
    rng = np.random.default_rng(np.random.SeedSequence([city_seed, draw]))
    return np.sort(pool[rng.choice(len(pool), BG_N, replace=False)])


def bg_cached(path, settings, compute):
    '''Load a cached result whose settings equal these, or compute, check and save it.'''
    if path.exists():
        with np.load(path, allow_pickle=False) as z:
            for k, v in settings.items():
                if not np.array_equal(z[k], v):
                    raise ValueError(f'I-16 cache {path.name}: {k} differs from this run. Delete the file to recompute.')
            print(f'  {path.name}: from the cache')
            return {k: z[k] for k in z.files}
    values, expected, fx, latency_ms = compute()
    result = {'shap_values': values, 'expected_value': np.float64(expected), 'prediction': fx,
              'efficiency_max_abs_error': np.float64(np.max(np.abs(values.sum(axis=1) + expected - fx))),
              'latency_ms': np.float64(latency_ms)}
    if not result['efficiency_max_abs_error'] < 1e-4:
        raise ValueError(f'I-16: {path.name}: values do not add up to the prediction')
    np.savez_compressed(path, **settings, **result)
    print(f'  {path.name}: computed ({latency_ms / 1000:.1f} s per window)')
    return {**settings, **result}


bg_trainer = CEEMDANFeatureTrainer.load(str(BG_MODEL), device=device)
with open(BG_SCALERS, 'rb') as f:
    bg_city_to_idx = pickle.load(f)['city_to_idx']
bg_model_sha = shap_file_sha256(BG_MODEL)
BG_RESULTS = {}   # city -> {'original': (n, F), 'draw1': ..., ...}, plus the backgrounds' dates
for city in BG_CITIES:
    print(f'{city}:')
    main = bg_main_result(city, bg_model_sha)
    seed = int(main['seed'])
    X_train_c, X_test_c = load_city_windows(BG_DATASET, city, bg_city_to_idx, bg_trainer)   # N2 guard
    train_dates, test_dates = shap_window_dates(BG_DATASET, city)
    in_test = np.isin(test_dates, main['test_dates'])
    if not np.array_equal(test_dates[in_test], main['test_dates']):
        raise ValueError(f'I-16: {city}: the explained test windows differ from the I-15 run')
    x = X_test_c[in_test]
    idx = bg_city_to_idx[city]

    def predict_fn(X, idx=idx):
        return bg_trainer.predict(X, np.full(len(X), idx, dtype=np.int64))

    # 1. the original background reproduces the saved values (first BG_N_REPRO windows)
    bg0 = X_train_c[np.isin(train_dates, main['background_dates'])]
    rep = bg_cached(BG_DIR / f'v2_{city}_lb{BG_LOOKBACK}_seed{seed}_repro{BG_N_REPRO}.npz',
                    {'city': np.array(city), 'seed': np.int64(seed), 'model_sha256': np.array(bg_model_sha),
                     'background_dates': main['background_dates'], 'test_dates': main['test_dates'][:BG_N_REPRO]},
                    lambda: shap_kernel_channel(predict_fn, bg0, x[:BG_N_REPRO], seed))
    rep_diff = float(np.max(np.abs(rep['shap_values'] - main['shap_values'][:BG_N_REPRO, 0, :])))
    if not rep_diff < 1e-4:
        raise ValueError(f'I-16: {city}: the original background does not reproduce the saved values ({rep_diff:.1e})')
    print(f'  original background reproduces the saved values of {BG_N_REPRO} windows (max diff {rep_diff:.1e})')

    # 2. new backgrounds, all 100 windows
    BG_RESULTS[city] = {'original': main['shap_values'][:, 0, :], 'dates_original': main['background_dates']}
    for draw in BG_DRAWS:
        bg_dates = bg_draw(train_dates, main['background_dates'], seed, draw)
        res = bg_cached(BG_DIR / f'v2_{city}_lb{BG_LOOKBACK}_n{BG_N_TEST}_bg{BG_N}_seed{seed}_draw{draw}.npz',
                        {'city': np.array(city), 'seed': np.int64(seed), 'draw': np.int64(draw),
                         'model_sha256': np.array(bg_model_sha), 'background_dates': bg_dates,
                         'test_dates': main['test_dates']},
                        lambda: shap_kernel_channel(predict_fn, X_train_c[np.isin(train_dates, bg_dates)], x, seed))
        if not np.allclose(res['prediction'], main['prediction'], atol=N2_PRED_ATOL):
            raise ValueError(f'I-16: {city}: predictions differ from the I-15 run')
        BG_RESULTS[city][f'draw{draw}'] = res['shap_values']
        BG_RESULTS[city][f'dates_draw{draw}'] = res['background_dates']
print('Done: run the next cell for the comparison.')


Delhi:
  v2_Delhi_lb7_seed133_repro10.npz: computed (7.9 s per window)
  original background reproduces the saved values of 10 windows (max diff 0.0e+00)
  v2_Delhi_lb7_n100_bg50_seed133_draw1.npz: computed (8.3 s per window)
  v2_Delhi_lb7_n100_bg50_seed133_draw2.npz: computed (8.3 s per window)
Mumbai:
  v2_Mumbai_lb7_seed143_repro10.npz: computed (8.3 s per window)
  original background reproduces the saved values of 10 windows (max diff 0.0e+00)
  v2_Mumbai_lb7_n100_bg50_seed143_draw1.npz: computed (8.3 s per window)
  v2_Mumbai_lb7_n100_bg50_seed143_draw2.npz: computed (8.3 s per window)
Bengaluru:
  v2_Bengaluru_lb7_seed127_repro10.npz: computed (8.3 s per window)
  original background reproduces the saved values of 10 windows (max diff 0.0e+00)
  v2_Bengaluru_lb7_n100_bg50_seed127_draw1.npz: computed (8.3 s per window)
  v2_Bengaluru_lb7_n100_bg50_seed127_draw2.npz: computed (8.3 s per window)
Done: run the next cell for the comparison.


In [16]:
# ── Revision 1, I-16: comparison of the backgrounds, per city ──
# Per background: mean |SHAP| per channel over the 100 windows, the decomposition share (IMF1 to slow), the IMF1 to
# IMF4 share, the slow channel's part of the decomposition share, the top 3 inputs, and against the original
# background the Spearman rank correlation of the 17 channels and the largest change of a channel's share (points).
BG_SUMMARY_CSV = BG_DIR / f'bg_check_summary_lb{BG_LOOKBACK}.csv'
rows = []
for city, res in BG_RESULTS.items():
    v0 = np.abs(res['original']).mean(axis=0)
    for name in ['original'] + [f'draw{d}' for d in BG_DRAWS]:
        v = np.abs(res[name]).mean(axis=0)
        dec = v[BG_N_POLLUTANTS:].sum()
        rho = spearmanr(v0, v)[0]
        rows.append({'city': city, 'background': name,
                     'overlap_with_original': int(np.isin(res[f'dates_{name}'], res['dates_original']).sum()),
                     'overlap_with_draw1': int(np.isin(res[f'dates_{name}'], res.get('dates_draw1', [])).sum()),
                     'decomposition_share_pct': 100 * dec / v.sum(),
                     'imf1_4_share_pct': 100 * v[BG_N_POLLUTANTS:-1].sum() / v.sum(),
                     'slow_of_decomposition_pct': 100 * v[-1] / dec,
                     'top3': ', '.join(BG_FEATURES[i] for i in np.argsort(-v)[:3]),
                     'spearman_rho_vs_original': rho,
                     'max_share_change_pts': 100 * float(np.max(np.abs(v / v.sum() - v0 / v0.sum()))),
                     **{f'mean_abs__{n}': x for n, x in zip(BG_FEATURES, v)}})
df_bg = pd.DataFrame(rows)
df_bg.to_csv(BG_SUMMARY_CSV, index=False)
show = ['city', 'background', 'overlap_with_original', 'overlap_with_draw1', 'decomposition_share_pct',
        'imf1_4_share_pct', 'slow_of_decomposition_pct', 'top3', 'spearman_rho_vs_original', 'max_share_change_pts']
print(df_bg[show].round(3).to_string(index=False))
print('\nRange over the backgrounds, per city:')
for city, g in df_bg.groupby('city', sort=False):
    print(f"  {city:10s} decomposition share {g.decomposition_share_pct.min():.1f} to {g.decomposition_share_pct.max():.1f}%, "
          f"IMF1-4 {g.imf1_4_share_pct.min():.2f} to {g.imf1_4_share_pct.max():.2f}%, lowest rho "
          f"{g.spearman_rho_vs_original.min():.3f}, same top 3 in every background: {g.top3.nunique() == 1}")
print(f'\nWritten: {BG_SUMMARY_CSV.name}')


     city background  overlap_with_original  overlap_with_draw1  decomposition_share_pct  imf1_4_share_pct  slow_of_decomposition_pct                      top3  spearman_rho_vs_original  max_share_change_pts
    Delhi   original                     50                   0                   23.983             3.267                     86.378 PM2.5, IMF5+ (slow), PM10                     1.000                 0.000
    Delhi      draw1                      0                  50                   23.924             3.308                     86.172 PM2.5, IMF5+ (slow), PM10                     0.978                 0.584
    Delhi      draw2                      0                   1                   23.114             3.344                     85.534 PM2.5, PM10, IMF5+ (slow)                     0.983                 3.207
   Mumbai   original                     50                   0                   17.657             2.537                     85.633 PM2.5, PM10, IMF5+ (slow)         